# ANN Pruning

This notebook implements pruning for Artificial Neural Networks (ANNs) by removing unused nodes with near-zero weights or activations. Supports both structured (node-level) and unstructured (weight-level) pruning.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import OrderedDict
from typing import List, Dict, Tuple, Optional, Union
import matplotlib.pyplot as plt
import copy

In [ ]:
class ANNPruner:
    """Prunes ANN by removing nodes/weights with near-zero magnitudes or activations"""
    
    def __init__(self, weight_threshold: float = 1e-6, activation_threshold: float = 1e-6):
        self.weight_threshold = weight_threshold
        self.activation_threshold = activation_threshold
        self.layer_stats = {}
        
    def analyze_weights(self, model: nn.Module) -> Dict[str, Dict]:
        """Analyze weight magnitudes across all layers"""
        weight_stats = {}
        
        for name, module in model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                weights = module.weight.data
                
                # Node-level analysis (average weight magnitude per output channel/neuron)
                if isinstance(module, nn.Conv2d):
                    # For conv layers: average over input channels, height, width
                    node_mags = torch.abs(weights).mean(dim=(1, 2, 3))
                else:
                    # For linear layers: average over input features
                    node_mags = torch.abs(weights).mean(dim=1)
                
                # Weight-level analysis (individual weight magnitudes)
                weight_mags = torch.abs(weights).flatten()
                
                weight_stats[name] = {
                    'weights': weights,
                    'node_magnitudes': node_mags,
                    'weight_magnitudes': weight_mags,
                    'mean_node_mag': node_mags.mean().item(),
                    'std_node_mag': node_mags.std().item(),
                    'min_node_mag': node_mags.min().item(),
                    'max_node_mag': node_mags.max().item(),
                    'node_zero_fraction': (node_mags < self.weight_threshold).float().mean().item(),
                    'weight_zero_fraction': (weight_mags < self.weight_threshold).float().mean().item(),
                    'layer_type': 'conv2d' if isinstance(module, nn.Conv2d) else 'linear',
                    'input_size': weights.shape[1:] if isinstance(module, nn.Conv2d) else weights.shape[1],
                    'output_size': weights.shape[0]
                }
                
        return weight_stats
    
    def find_prunable_nodes(self, weight_stats: Dict[str, Dict]) -> Dict[str, List[int]]:
        """Find nodes that can be pruned based on weight magnitudes"""
        prunable_nodes = {}
        
        for layer_name, stats in weight_stats.items():
            node_magnitudes = stats['node_magnitudes']
            prunable_mask = node_magnitudes < self.weight_threshold
            prunable_indices = torch.where(prunable_mask)[0].tolist()
            
            if prunable_indices:
                prunable_nodes[layer_name] = prunable_indices
                
        return prunable_nodes
    
    def find_prunable_weights(self, weight_stats: Dict[str, Dict], sparsity_ratio: float = 0.5) -> Dict[str, torch.Tensor]:
        """Find individual weights to prune for unstructured pruning"""
        prunable_weights = {}
        
        for layer_name, stats in weight_stats.items():
            weight_magnitudes = stats['weight_magnitudes']
            weights_shape = stats['weights'].shape
            
            # Sort weights by magnitude and select bottom percentile
            num_weights_to_prune = int(len(weight_magnitudes) * sparsity_ratio)
            _, indices = torch.topk(weight_magnitudes, num_weights_to_prune, largest=False)
            
            # Convert flat indices to multi-dimensional indices
            mask = torch.zeros_like(weight_magnitudes, dtype=torch.bool)
            mask[indices] = True
            mask = mask.reshape(weights_shape)
            
            prunable_weights[layer_name] = mask
                
        return prunable_weights
    
    def collect_activations(self, model: nn.Module, test_data: torch.Tensor) -> Dict[str, torch.Tensor]:
        """Collect activations from test data to identify unused nodes"""
        activations = {}
        hooks = []
        
        def make_hook(name):
            def hook(module, input, output):
                if name not in activations:
                    activations[name] = []
                activations[name].append(output.detach().cpu())
            return hook
        
        # Register hooks for ReLU activations and conv/linear outputs
        for name, module in model.named_modules():
            if isinstance(module, (nn.ReLU, nn.Conv2d, nn.Linear)):
                hook = module.register_forward_hook(make_hook(name))
                hooks.append(hook)
        
        model.eval()
        with torch.no_grad():
            for batch in test_data:
                if isinstance(batch, (tuple, list)):
                    batch = batch[0]
                _ = model(batch)
        
        # Remove hooks
        for hook in hooks:
            hook.remove()
            
        # Concatenate collected activations
        for name in activations:
            activations[name] = torch.cat(activations[name], dim=0)
            
        return activations
    
    def analyze_activations(self, activations: Dict[str, torch.Tensor]) -> Dict[str, Dict]:
        """Analyze activation patterns to find unused nodes"""
        activation_stats = {}
        
        for name, acts in activations.items():
            # For each output channel/neuron, compute average activation
            if len(acts.shape) == 4:  # Conv layer output (N, C, H, W)
                channel_acts = acts.mean(dim=(0, 2, 3))  # Average over batch, height, width
            elif len(acts.shape) == 2:  # FC layer output (N, C)
                channel_acts = acts.mean(dim=0)  # Average over batch
            else:
                continue
                
            activation_stats[name] = {
                'activations': channel_acts,
                'mean_act': channel_acts.mean().item(),
                'std_act': channel_acts.std().item(),
                'min_act': channel_acts.min().item(),
                'max_act': channel_acts.max().item(),
                'zero_fraction': (channel_acts < self.activation_threshold).float().mean().item(),
                'dead_neurons': torch.sum(channel_acts == 0).item(),
                'total_neurons': len(channel_acts)
            }
            
        return activation_stats
    
    def find_inactive_nodes(self, activation_stats: Dict[str, Dict]) -> Dict[str, List[int]]:
        """Find nodes with near-zero activations (dead neurons)"""
        inactive_nodes = {}
        
        for layer_name, stats in activation_stats.items():
            activations = stats['activations']
            inactive_mask = activations < self.activation_threshold
            inactive_indices = torch.where(inactive_mask)[0].tolist()
            
            if inactive_indices:
                inactive_nodes[layer_name] = inactive_indices
                
        return inactive_nodes

In [ ]:
    def apply_unstructured_pruning(self, model: nn.Module, prunable_weights: Dict[str, torch.Tensor]) -> nn.Module:
        """Apply unstructured pruning by zeroing out selected weights"""
        pruned_model = copy.deepcopy(model)
        
        for name, module in pruned_model.named_modules():
            if name in prunable_weights and isinstance(module, (nn.Conv2d, nn.Linear)):
                mask = prunable_weights[name]
                # Zero out pruned weights
                module.weight.data[mask] = 0.0
                
                # Optionally zero out corresponding biases for conv layers
                if isinstance(module, nn.Conv2d) and module.bias is not None:
                    # For conv layers, zero bias if all weights for that channel are pruned
                    channel_mask = mask.any(dim=(1, 2, 3))  # Any weight in channel is pruned
                    if channel_mask.all():  # If all weights in a channel are pruned
                        module.bias.data[channel_mask] = 0.0
        
        return pruned_model
    
    def create_pruned_model(self, original_model: nn.Module, prunable_nodes: Dict[str, List[int]]) -> nn.Module:
        """Create a new model with pruned nodes removed (structured pruning)"""
        # Calculate new dimensions after pruning
        new_dims = self._calculate_new_dimensions(original_model, prunable_nodes)
        
        # Create new model architecture
        pruned_model = self._build_pruned_architecture(original_model, new_dims)
        
        # Copy weights excluding pruned nodes
        self._copy_pruned_weights(original_model, pruned_model, prunable_nodes)
        
        return pruned_model
    
    def _calculate_new_dimensions(self, model: nn.Module, prunable_nodes: Dict[str, List[int]]) -> Dict[str, Tuple]:
        """Calculate new layer dimensions after pruning"""
        new_dims = {}
        
        for name, module in model.named_modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                if name in prunable_nodes:
                    pruned_count = len(prunable_nodes[name])
                    if isinstance(module, nn.Conv2d):
                        new_out_channels = module.out_channels - pruned_count
                        new_dims[name] = (module.in_channels, new_out_channels, 
                                        module.kernel_size, module.stride, module.padding)
                    else:  # Linear
                        new_out_features = module.out_features - pruned_count
                        new_dims[name] = (module.in_features, new_out_features)
                else:
                    if isinstance(module, nn.Conv2d):
                        new_dims[name] = (module.in_channels, module.out_channels,
                                        module.kernel_size, module.stride, module.padding)
                    else:  # Linear
                        new_dims[name] = (module.in_features, module.out_features)
        
        return new_dims
    
    def _build_pruned_architecture(self, original_model: nn.Module, new_dims: Dict[str, Tuple]) -> nn.Module:
        """Build new model architecture with updated dimensions"""
        # This assumes the PongQNetwork architecture from the provided code
        # For a general solution, this would need to be more flexible
        
        class PrunedPongQNetwork(nn.Module):
            def __init__(self, conv_dims, fc_dims):
                super().__init__()
                
                # Convolutional layers
                conv1_in, conv1_out, k1, s1, p1 = conv_dims.get('conv1', (4, 32, (8, 8), (4, 4), (0, 0)))
                conv2_in, conv2_out, k2, s2, p2 = conv_dims.get('conv2', (32, 64, (4, 4), (2, 2), (0, 0)))
                conv3_in, conv3_out, k3, s3, p3 = conv_dims.get('conv3', (64, 64, (3, 3), (1, 1), (0, 0)))
                
                self.conv1 = nn.Conv2d(conv1_in, conv1_out, kernel_size=k1, stride=s1, padding=p1)
                self.conv2 = nn.Conv2d(conv2_out, conv2_out, kernel_size=k2, stride=s2, padding=p2)  # Note: input adjusted
                self.conv3 = nn.Conv2d(conv2_out, conv3_out, kernel_size=k3, stride=s3, padding=p3)
                
                # Calculate conv output size: conv3_out * 7 * 7 (based on original architecture)
                conv_output_size = conv3_out * 7 * 7
                
                # Linear layers
                fc1_in, fc1_out = fc_dims.get('fc1', (conv_output_size, 512))
                fc2_in, fc2_out = fc_dims.get('fc2', (512, 6))
                
                self.fc1 = nn.Linear(fc1_in, fc1_out)
                self.fc2 = nn.Linear(fc1_out, fc2_out)  # Note: input adjusted
                
            def forward(self, x):
                x = F.relu(self.conv1(x))
                x = F.relu(self.conv2(x))
                x = F.relu(self.conv3(x))
                x = x.view(x.size(0), -1)  # Flatten
                x = F.relu(self.fc1(x))
                x = self.fc2(x)
                return x
        
        # Separate conv and fc dimensions
        conv_dims = {k: v for k, v in new_dims.items() if len(v) == 5}
        fc_dims = {k: v for k, v in new_dims.items() if len(v) == 2}
        
        return PrunedPongQNetwork(conv_dims, fc_dims)
    
    def _copy_pruned_weights(self, original_model: nn.Module, pruned_model: nn.Module, 
                           prunable_nodes: Dict[str, List[int]]):
        """Copy weights from original to pruned model, excluding pruned nodes"""
        orig_dict = dict(original_model.named_modules())
        pruned_dict = dict(pruned_model.named_modules())
        
        for name in orig_dict:
            if name in pruned_dict and isinstance(orig_dict[name], (nn.Conv2d, nn.Linear)):
                orig_module = orig_dict[name]
                pruned_module = pruned_dict[name]
                
                pruned_indices = prunable_nodes.get(name, [])
                
                if isinstance(orig_module, nn.Conv2d):
                    self._copy_conv_weights(orig_module, pruned_module, pruned_indices)
                elif isinstance(orig_module, nn.Linear):
                    self._copy_linear_weights(orig_module, pruned_module, pruned_indices)
    
    def _copy_conv_weights(self, orig_conv: nn.Conv2d, pruned_conv: nn.Conv2d, pruned_indices: List[int]):
        """Copy conv weights excluding pruned output channels"""
        with torch.no_grad():
            # Create mask for keeping channels
            all_indices = set(range(orig_conv.out_channels))
            keep_indices = sorted(list(all_indices - set(pruned_indices)))
            
            # Copy weights
            pruned_conv.weight.data = orig_conv.weight.data[keep_indices]
            
            # Copy bias if present
            if orig_conv.bias is not None and pruned_conv.bias is not None:
                pruned_conv.bias.data = orig_conv.bias.data[keep_indices]
    
    def _copy_linear_weights(self, orig_linear: nn.Linear, pruned_linear: nn.Linear, pruned_indices: List[int]):
        """Copy linear weights excluding pruned output features"""
        with torch.no_grad():
            # Create mask for keeping features
            all_indices = set(range(orig_linear.out_features))
            keep_indices = sorted(list(all_indices - set(pruned_indices)))
            
            # Copy weights
            pruned_linear.weight.data = orig_linear.weight.data[keep_indices]
            
            # Copy bias if present
            if orig_linear.bias is not None and pruned_linear.bias is not None:
                pruned_linear.bias.data = orig_linear.bias.data[keep_indices]

In [ ]:
def load_and_prune_ann(model_path: str, test_data_loader=None, 
                      weight_threshold: float = 1e-6, 
                      activation_threshold: float = 1e-6,
                      pruning_type: str = 'structured',
                      sparsity_ratio: float = 0.5,
                      use_activation_pruning: bool = True) -> Tuple[nn.Module, Dict]:
    """Complete pipeline to load, analyze, and prune an ANN model
    
    Args:
        model_path: Path to the model file
        test_data_loader: DataLoader for activation analysis
        weight_threshold: Threshold for weight-based pruning
        activation_threshold: Threshold for activation-based pruning
        pruning_type: 'structured' (node removal) or 'unstructured' (weight zeroing)
        sparsity_ratio: Fraction of weights to prune for unstructured pruning
        use_activation_pruning: Whether to use activation analysis
    """
    
    # Load model
    print(f"Loading model from {model_path}...")
    model = torch.load(model_path, map_location='cpu')
    model.eval()
    
    print(f"Original model architecture:")
    print(model)
    
    # Initialize pruner
    pruner = ANNPruner(weight_threshold=weight_threshold, 
                      activation_threshold=activation_threshold)
    
    # Analyze weights
    print("\nAnalyzing weights...")
    weight_stats = pruner.analyze_weights(model)
    
    print("Weight analysis results:")
    for layer_name, stats in weight_stats.items():
        print(f"  {layer_name}: {stats['output_size']} nodes, "
              f"{stats['node_zero_fraction']:.1%} near-zero nodes, "
              f"{stats['weight_zero_fraction']:.1%} near-zero weights")
    
    # Find prunable elements
    if pruning_type == 'structured':
        weight_prunable = pruner.find_prunable_nodes(weight_stats)
        print(f"\nWeight-based prunable nodes: {weight_prunable}")
    else:  # unstructured
        weight_prunable = pruner.find_prunable_weights(weight_stats, sparsity_ratio)
        print(f"\nUnstructured pruning will remove {sparsity_ratio:.1%} of weights in each layer")
    
    # Analyze activations if test data provided
    activation_prunable = {}
    if use_activation_pruning and test_data_loader is not None:
        print("\nAnalyzing activations...")
        activations = pruner.collect_activations(model, test_data_loader)
        activation_stats = pruner.analyze_activations(activations)
        
        print("Activation analysis results:")
        for layer_name, stats in activation_stats.items():
            print(f"  {layer_name}: {stats['dead_neurons']}/{stats['total_neurons']} "
                  f"dead neurons ({stats['zero_fraction']:.1%})")
        
        if pruning_type == 'structured':
            activation_prunable = pruner.find_inactive_nodes(activation_stats)
            print(f"\nActivation-based prunable nodes: {activation_prunable}")
    
    # Apply pruning
    if pruning_type == 'structured':
        # Combine pruning criteria for structured pruning
        all_prunable = {}
        for layer in set(list(weight_prunable.keys()) + list(activation_prunable.keys())):
            weight_nodes = set(weight_prunable.get(layer, []))
            activation_nodes = set(activation_prunable.get(layer, []))
            # Take intersection for conservative pruning
            combined_nodes = weight_nodes.intersection(activation_nodes) if activation_prunable else weight_nodes
            if combined_nodes:
                all_prunable[layer] = sorted(list(combined_nodes))
        
        print(f"\nFinal prunable nodes: {all_prunable}")
        
        if all_prunable:
            print("\nCreating structurally pruned model...")
            pruned_model = pruner.create_pruned_model(model, all_prunable)
        else:
            print("\nNo nodes found for structured pruning.")
            pruned_model = model
            
    else:  # unstructured
        print("\nApplying unstructured pruning...")
        pruned_model = pruner.apply_unstructured_pruning(model, weight_prunable)
        all_prunable = weight_prunable
    
    # Calculate statistics
    orig_params = sum(p.numel() for p in model.parameters())
    pruned_params = sum(p.numel() for p in pruned_model.parameters())
    
    if pruning_type == 'unstructured':
        # For unstructured pruning, count non-zero parameters
        non_zero_params = sum((p != 0).sum().item() for p in pruned_model.parameters())
        effective_compression = non_zero_params / orig_params
    else:
        effective_compression = pruned_params / orig_params
    
    stats = {
        'original_parameters': orig_params,
        'pruned_parameters': pruned_params,
        'effective_parameters': non_zero_params if pruning_type == 'unstructured' else pruned_params,
        'compression_ratio': effective_compression,
        'parameter_reduction': 1 - effective_compression,
        'pruning_type': pruning_type,
        'prunable_elements': all_prunable,
        'weight_stats': weight_stats,
        'sparsity_ratio': sparsity_ratio if pruning_type == 'unstructured' else None
    }
    
    print(f"\n{pruning_type.title()} Pruning Statistics:")
    print(f"Original parameters: {orig_params:,}")
    if pruning_type == 'structured':
        print(f"Pruned parameters: {pruned_params:,}")
    else:
        print(f"Pruned parameters: {pruned_params:,} (structure unchanged)")
        print(f"Non-zero parameters: {non_zero_params:,}")
    print(f"Effective compression ratio: {effective_compression:.3f}")
    print(f"Parameter reduction: {1-effective_compression:.1%}")
    
    return pruned_model, stats

In [ ]:
def compare_models(original_model: nn.Module, pruned_model: nn.Module, test_data: torch.Tensor) -> Dict:
    """Compare performance between original and pruned models"""
    
    def evaluate_model(model, data):
        model.eval()
        outputs = []
        with torch.no_grad():
            for batch in data:
                if isinstance(batch, (tuple, list)):
                    batch = batch[0]
                output = model(batch)
                outputs.append(output)
        return torch.cat(outputs, dim=0)
    
    orig_outputs = evaluate_model(original_model, test_data)
    pruned_outputs = evaluate_model(pruned_model, test_data)
    
    # Calculate differences
    mse = F.mse_loss(pruned_outputs, orig_outputs).item()
    mae = F.l1_loss(pruned_outputs, orig_outputs).item()
    
    # Calculate cosine similarity
    cos_sim = F.cosine_similarity(orig_outputs.flatten(), pruned_outputs.flatten(), dim=0).item()
    
    return {
        'mse': mse,
        'mae': mae,
        'cosine_similarity': cos_sim,
        'max_difference': (pruned_outputs - orig_outputs).abs().max().item()
    }

In [ ]:
def visualize_ann_pruning_stats(stats: Dict):
    """Visualize ANN pruning statistics"""
    if not stats.get('weight_stats'):
        print("No weight statistics to visualize")
        return
    
    weight_stats = stats['weight_stats']
    pruning_type = stats.get('pruning_type', 'structured')
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'ANN {pruning_type.title()} Pruning Analysis', fontsize=16)
    
    layer_names = list(weight_stats.keys())
    
    # Plot 1: Node magnitude distributions
    mean_node_mags = [weight_stats[name]['mean_node_mag'] for name in layer_names]
    node_zero_fractions = [weight_stats[name]['node_zero_fraction'] for name in layer_names]
    
    axes[0, 0].bar(range(len(layer_names)), mean_node_mags, color='skyblue')
    axes[0, 0].set_title('Mean Node Weight Magnitudes')
    axes[0, 0].set_xlabel('Layer')
    axes[0, 0].set_ylabel('Mean Magnitude')
    axes[0, 0].set_xticks(range(len(layer_names)))
    axes[0, 0].set_xticklabels([name.split('.')[-1] for name in layer_names], rotation=45)
    
    # Plot 2: Zero fraction comparison
    weight_zero_fractions = [weight_stats[name]['weight_zero_fraction'] for name in layer_names]
    
    x = np.arange(len(layer_names))
    width = 0.35
    
    axes[0, 1].bar(x - width/2, node_zero_fractions, width, label='Node-level', color='lightcoral')
    axes[0, 1].bar(x + width/2, weight_zero_fractions, width, label='Weight-level', color='lightblue')
    axes[0, 1].set_title('Fraction of Near-Zero Elements')
    axes[0, 1].set_xlabel('Layer')
    axes[0, 1].set_ylabel('Zero Fraction')
    axes[0, 1].set_xticks(x)
    axes[0, 1].set_xticklabels([name.split('.')[-1] for name in layer_names], rotation=45)
    axes[0, 1].legend()
    
    # Plot 3: Weight distribution for first layer
    if layer_names:
        first_layer = layer_names[0]
        weight_mags = weight_stats[first_layer]['weight_magnitudes'].numpy()
        
        axes[0, 2].hist(weight_mags, bins=50, alpha=0.7, color='green')
        axes[0, 2].axvline(x=stats.get('weight_threshold', 1e-6), color='r', 
                          linestyle='--', label='Pruning Threshold')
        axes[0, 2].set_title(f'Weight Distribution - {first_layer.split(".")[-1]}')
        axes[0, 2].set_xlabel('Weight Magnitude')
        axes[0, 2].set_ylabel('Count')
        axes[0, 2].legend()
        axes[0, 2].set_yscale('log')
    
    # Plot 4: Layer sizes
    layer_sizes = [weight_stats[name]['output_size'] for name in layer_names]
    
    axes[1, 0].bar(range(len(layer_names)), layer_sizes, color='orange')
    axes[1, 0].set_title('Layer Output Sizes')
    axes[1, 0].set_xlabel('Layer')
    axes[1, 0].set_ylabel('Output Size')
    axes[1, 0].set_xticks(range(len(layer_names)))
    axes[1, 0].set_xticklabels([name.split('.')[-1] for name in layer_names], rotation=45)
    
    # Plot 5: Compression visualization
    if 'compression_ratio' in stats:
        if pruning_type == 'structured':
            labels = ['Kept Parameters', 'Removed Parameters']
            sizes = [stats['compression_ratio'], 1 - stats['compression_ratio']]
        else:
            kept_ratio = stats['effective_parameters'] / stats['original_parameters']
            labels = ['Non-zero Weights', 'Zeroed Weights']
            sizes = [kept_ratio, 1 - kept_ratio]
        
        colors = ['lightblue', 'lightcoral']
        axes[1, 1].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
        axes[1, 1].set_title(f'{pruning_type.title()} Pruning Results')
    else:
        axes[1, 1].text(0.5, 0.5, 'No Pruning Applied', 
                       horizontalalignment='center', verticalalignment='center',
                       transform=axes[1, 1].transAxes, fontsize=14)
        axes[1, 1].set_title('Pruning Status')
    
    # Plot 6: Sparsity pattern (for unstructured pruning)
    if pruning_type == 'unstructured' and stats.get('sparsity_ratio'):
        sparsity = stats['sparsity_ratio']
        axes[1, 2].bar(['Target Sparsity'], [sparsity], color='purple', alpha=0.7)
        axes[1, 2].set_title('Target Sparsity Ratio')
        axes[1, 2].set_ylabel('Sparsity Ratio')
        axes[1, 2].set_ylim(0, 1)
    else:
        # Show pruned nodes per layer for structured pruning
        prunable_elements = stats.get('prunable_elements', {})
        if isinstance(prunable_elements, dict) and any(isinstance(v, list) for v in prunable_elements.values()):
            pruned_counts = [len(prunable_elements.get(name, [])) for name in layer_names]
            axes[1, 2].bar(range(len(layer_names)), pruned_counts, color='red', alpha=0.7)
            axes[1, 2].set_title('Pruned Nodes per Layer')
            axes[1, 2].set_xlabel('Layer')
            axes[1, 2].set_ylabel('Pruned Nodes')
            axes[1, 2].set_xticks(range(len(layer_names)))
            axes[1, 2].set_xticklabels([name.split('.')[-1] for name in layer_names], rotation=45)
        else:
            axes[1, 2].text(0.5, 0.5, 'No Structured\nPruning Data', 
                           horizontalalignment='center', verticalalignment='center',
                           transform=axes[1, 2].transAxes, fontsize=12)
    
    plt.tight_layout()
    plt.show()

## Usage Examples

In [ ]:
# Example 1: Structured pruning (node removal)
# model_path = "path/to/your/ann_model.pth"
# 
# pruned_model, stats = load_and_prune_ann(
#     model_path,
#     weight_threshold=1e-6,
#     pruning_type='structured',
#     use_activation_pruning=False
# )
# 
# print(f"\nOriginal model:")
# print(original_model)
# print(f"\nPruned model:")
# print(pruned_model)
# 
# # Visualize results
# visualize_ann_pruning_stats(stats)
# 
# # Save pruned model
# torch.save(pruned_model, "pruned_ann_structured.pth")

In [ ]:
# Example 2: Unstructured pruning (weight zeroing)
# 
# pruned_model, stats = load_and_prune_ann(
#     model_path,
#     pruning_type='unstructured',
#     sparsity_ratio=0.5,  # Remove 50% of weights
#     use_activation_pruning=False
# )
# 
# # Check sparsity
# total_params = sum(p.numel() for p in pruned_model.parameters())
# zero_params = sum((p == 0).sum().item() for p in pruned_model.parameters())
# actual_sparsity = zero_params / total_params
# print(f"Achieved sparsity: {actual_sparsity:.1%}")
# 
# # Visualize results
# visualize_ann_pruning_stats(stats)
# 
# # Save pruned model
# torch.save(pruned_model, "pruned_ann_unstructured.pth")

In [ ]:
# Example 3: Combined weight and activation-based pruning
# 
# # Create test data loader (replace with your actual test data)
# # test_data = [torch.randn(32, 4, 84, 84) for _ in range(10)]  # Example test batches
# 
# pruned_model, stats = load_and_prune_ann(
#     model_path,
#     test_data_loader=test_data,
#     weight_threshold=1e-6,
#     activation_threshold=1e-6,
#     pruning_type='structured',
#     use_activation_pruning=True
# )
# 
# # Compare model outputs
# original_model = torch.load(model_path, map_location='cpu')
# comparison = compare_models(original_model, pruned_model, test_data[:5])  # Use subset for comparison
# 
# print(f"\nModel Comparison:")
# print(f"MSE: {comparison['mse']:.6f}")
# print(f"MAE: {comparison['mae']:.6f}")
# print(f"Cosine Similarity: {comparison['cosine_similarity']:.6f}")
# print(f"Max Difference: {comparison['max_difference']:.6f}")
# 
# # Visualize results
# visualize_ann_pruning_stats(stats)

## Notes

**Structured vs Unstructured Pruning:**
- **Structured**: Removes entire nodes/channels, reducing model size
- **Unstructured**: Zeros individual weights, maintains architecture but creates sparse weights

**Key Features:**
- Supports both Conv2d and Linear layers
- Weight-based and activation-based pruning criteria
- Model comparison tools
- Comprehensive visualization

**Architecture Support:**
Currently optimized for the PongQNetwork architecture but can be extended for other architectures by modifying the `_build_pruned_architecture` method.